# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields, and following FAIR principles for data interaction.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant dataset (metadata only)
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

# Print dataset name and description
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review record sets and fields available in the FAIR² dataset, referencing entities by their `@id`. The dataset may contain multiple record sets, each identified by a unique `@id`. We will enumerate the top-level record sets and their referenced fields.

In [ ]:
# List all record sets and fields by their @id

record_sets = list(md.record_sets)
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}  name: {rs.get('name', '')}")
    # List fields by @id
    if 'field' in rs:
        # It's possible for 'field' to be a dict or list
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '')}")
    print("")

## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis. All record set and field references are by their `@id`.

In [ ]:
# Extract data from all record sets by @id and load into pandas DataFrames

# Get all RecordSet @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    # Try to load records for this record set (skip if error)
    try:
        df = pd.DataFrame(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Choose the main record set (if only one, pick it)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in '{main_record_set_id}': \n{dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()
else:
    print("No record sets with records found.")

## 4. Exploratory Data Analysis (EDA)
Example filtering, normalization, and aggregation operations are performed on fields using their `@id`. Here we select a numeric field (using its `@id`), filter for values above a threshold, normalize the column, and aggregate by a group field if present.

In [ ]:
# ---- EDA ----
# Adjust these IDs based on actual field names seen previously
# Here we try to pick a likely numeric field and group field from columns

df = dataframes[main_record_set_id]
numeric_field_candidates = [c for c in df.columns if c.lower() in ('coverage', 'mw', 'molecular_weight', 'abundance', 'peptide_count', 'unique_peptides', 'score')]
group_field_candidates = [c for c in df.columns if c.lower() in ('sample', 'condition', 'group', 'accession', 'id')]

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Selected numeric field for analysis: {numeric_field_id}")
else:
    numeric_field_id = df.columns[df.dtypes == 'float64'][0] if any(df.dtypes == 'float64') else df.columns[0]
    print(f"No standard fields found; using: {numeric_field_id}")

threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df[[numeric_field_id]].head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If a useful group field exists
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
    print(f"\nGrouped data by {group_field_id} (showing group means):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field and its normalized values, or compare groups if relevant. Uses matplotlib for a basic plot.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the normalized numeric field
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].hist(filtered_df[numeric_field_id], bins=30, alpha=0.8)
axs[0].set_title(f"Distribution of {numeric_field_id}")
axs[0].set_xlabel(numeric_field_id)
axs[0].set_ylabel("Count")

axs[1].hist(filtered_df[f"{numeric_field_id}_normalized"], bins=30, alpha=0.8, color='orange')
axs[1].set_title(f"Distribution of normalized {numeric_field_id}")
axs[1].set_xlabel(f"{numeric_field_id}_normalized")
axs[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

# If group_field_id exists, plot boxplots for groups
if 'group_field_id' in locals():
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, rot=45, figsize=(8,5))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.show()

## 6. Conclusion
In this notebook, we have demonstrated how to:
- Load and inspect a machine-actionable FAIR² dataset using `mlcroissant`
- Reference all dataset components by their `@id`
- Extract record sets, explore fields, and perform EDA with filtering, normalization, and grouping
- Visualize key numeric data and its distribution

Continue your analysis by exploring additional record sets, linking records by ID, and building reproducible pipelines with Croissant schemas. See [mlcroissant documentation](https://mlcommons.github.io/croissant/) for further details.